# ДЗ4 (додатково). Порівняння Gemini і GPT-4.1-nano на власних промптах

Використав **Vertex AI**, бо на Google Cloud був хакатон-кредит (гроші вже
були на балансі), тож підключив кілька моделей **Gemini** саме через Vertex.
Для порівняння додав ще й **OpenAI** (`gpt-4.1-nano`) напряму, щоб
подивитись, як різні провайдери реагують на ті самі промпти.

Мета цього ноутбука не в тому, щоб просто підібрати промпти, які дають
правильну відповідь — це можна зробити під одну конкретну модель. Мета —
подивитись, як **різні моделі поводяться під одними й тими самими
промптами**: де вони збігаються, а де розходяться, і чому.

## Як запустити

Цей ноутбук розрахований на **Google Colab**, не на локальний запуск:

1. Відкрий файл у [Google Colab](https://colab.research.google.com/).
2. Виконай комірки по порядку зверху вниз.
3. У другій комірці (`auth.authenticate_user()`) з'явиться спливне вікно
   Google, залогинься тим самим акаунтом, яким керуєш проєктом
   `neoversity-llm-compare` (з хакатон-кредитом $150). Ключ сервісного
   акаунта тут навмисно не використовується.
4. У тій самій комірці Colab запитає `OPENAI_API_KEY` через окреме поле
   `getpass` — встав туди свій ключ OpenAI (не в код!). Він ніде не
   зберігається, не потрапляє у файл і зникає, коли завершується сесія
   Colab.
5. `LOCATION = "global"` навмисно, бо не всі моделі Gemini доступні в
   регіональних endpoint.
6. Якщо Gemini-модель поверне 404 "not found" — перевір точну назву
   моделі в Model Garden твого проєкту
   (`console.cloud.google.com/agent-platform/model-garden`).

## Комірка 0. Встановлення бібліотеки та авторизація

`google-genai` це офіційна бібліотека Google для виклику Gemini як через
звичайний AI Studio API, так і через Vertex AI (параметр `vertexai=True`).

In [ ]:
!pip install -q google-genai openai

import getpass
from google.colab import auth

auth.authenticate_user()
OPENAI_API_KEY = getpass.getpass("OPENAI_API_KEY: ")
print("Авторизація Google пройдена, ключ OpenAI отримано")

In [ ]:
PROJECT_ID = "neoversity-llm-compare"  # проєкт з хакатон-кредитом
LOCATION = "global"  # партнерські й деякі Google-моделі доступні лише через global

from google import genai
from openai import OpenAI

gemini_client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# (provider, model_id) — два різні провайдери в одному списку порівняння
MODELS = [
    ("gemini", "gemini-2.5-flash-lite"),   # найдешевша й найпростіша з Gemini
    ("gemini", "gemini-3.1-pro-preview"),  # флагманська Pro-модель Google
    ("openai", "gpt-4.1-nano"),            # найменша модель OpenAI, напряму, не через Vertex
]


def ask_model(provider: str, model: str, system_prompt: str, user_message: str) -> str:
    if provider == "gemini":
        resp = gemini_client.models.generate_content(
            model=model,
            contents=user_message,
            config={"system_instruction": system_prompt.strip(), "temperature": 0},
        )
        return resp.text.strip()
    if provider == "openai":
        resp = openai_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "developer", "content": system_prompt.strip()},
                {"role": "user", "content": user_message},
            ],
            temperature=0,
        )
        return resp.choices[0].message.content.strip()
    raise ValueError(f"Невідомий провайдер: {provider}")

## Комірка 1. Ті самі три завдання, ті самі промпти

Тексти й промпти скопійовані один в один з основного файлу ДЗ4, щоб
порівняння було чесним: одна й та сама інструкція, різні моделі.

In [ ]:
texts_1 = [
    "Оксана Дрозд натхненно провела демо — команда аплодувала стоячи.",
    "Керівник відділу Андрій Король грубо порушив інструкції, через що реліз зірвано.",
    "Роман Бондаренко надіслав звіт о 14:00 і підтвердив графік зустрічей.",
    "Сьогодні Ілля Ткачук блискуче оптимізував код і заощадив 30% бюджету.",
    "Марко Савченко ігнорує повідомлення й затримує погодження — це неприйнятно.",
]

texts_2 = [
    "Квитанції не завантажуються в Atlas Pro з мобільного — фінвідділ стоїть.",
    "Після оновлення 3.1 у Core графіки будуються на 2 секунди довше, але працюють стабільно.",
    "Автовідповідь у Desk на свята — nice-to-have, допоможе трохи зменшити навантаження на операторів.",
    "Core падає під час імпорту CSV >100 тис. рядків — роботу заблоковано.",
    "Atlas показує старі курси валют, поки що оновлюємо вручну.",
]

texts_3 = [
    "Вийшла версія 2.0 з офлайн-режимом і новим дизайном.",
    "У частини користувачів не відкривається профіль після апдейту; працюємо над виправленням.",
    "Збираємо відгуки про новий модуль аналітики.",
    "На вихідних сервіс у режимі планового обслуговування.",
    "Рекомендуємо вмикати двофакторну автентифікацію.",
]

PROMPT_STUDENT_1 = """
Проаналізуй речення, знайди ім'я та прізвище, і тональність відповіді, яка
очікується — вона може бути лише позитивна|нейтральна|негативна.
Ти маєш дати відповідь тільки в такій формі:
{"person":"<повне ім'я з тексту>", "tone":"позитивна|нейтральна|негативна"}
Приклад
Текст: <<<Керівник проєкту Ігор Лисенко надіслав протокол і закрив тікет о 10:15.>>>
{"person":"Ігор Лисенко","tone":"нейтральна"}
Ти не маєш права щось додавати, форма має строго відповідати!
"""

PROMPT_STUDENT_2 = """
Знайди в тексті назву продукту, потім знайди пріоритет.
Назва продукту має бути лише базовою формою: якщо в тексті є "Atlas Pro",
"Atlas Mobile", "Core 3.1", "Desk Cloud" тощо — прибирай версію, редакцію чи
платформу і залишай тільки "Atlas", "Core" або "Desk".
Формат відповіді: модель повертає тільки JSON з рівно двома ключами.
Проаналізуй пріоритет, враховуючи, наскільки критична проблема.
Якщо процес зупиняється — це високий пріоритет.
Якщо дані невірні — це також високий пріоритет.
Якщо є затримка в часі, але все працює — це середній пріоритет.
Пріоритет може бути лише (високий|середній|низький).
Ти маєш дати відповідь тільки в такій формі:
{"product":"Atlas|Core|Desk","priority":"високий|середній|низький"}
Строго дотримуйся формату виводу: лише сам JSON, без жодних пояснень, без
markdown, без потрійних лапок ```json навколо відповіді.
Приклад:
{"product":"Atlas","priority":"високий"}
Відповідь має бути тільки JSON з рівно двома ключами.
"""

PROMPT_STUDENT_3 = """
Знайди в тексті стадію розробки ПЗ.
Стадія може бути лише категорією з множини реліз | інцидент | рекомендація.
Також ти повинен створити заголовок, який описує подію. Він має бути
коротшим за 8 слів.

Ти маєш дати відповідь тільки в такій формі:
{"title":"<до 8 слів>","category":"реліз|інцидент|рекомендація"}
Строго дотримуйся формату виводу: лише сам JSON, без жодних пояснень, без
markdown, без потрійних лапок ```json навколо відповіді.
Приклад:
{"title":"Версія 2.0 з офлайн‑режимом та новим дизайном","category":"реліз"}
Відповідь має бути тільки JSON з рівно двома ключами.
"""

TASKS = [
    ("Завдання 1 (person+tone)", PROMPT_STUDENT_1, texts_1,
     "Проаналізуй текст і поверни ТІЛЬКИ JSON з ключами person та tone.\nТекст: <<<{t}>>>"),
    ("Завдання 2 (product+priority)", PROMPT_STUDENT_2, texts_2,
     "Проаналізуй відгук і поверни ТІЛЬКИ JSON з ключами product та priority.\nТекст: <<<{t}>>>"),
    ("Завдання 3 (title+category)", PROMPT_STUDENT_3, texts_3,
     "Склади заголовок і визнач категорію. Поверни ТІЛЬКИ JSON з ключами title та category.\nТекст: <<<{t}>>>"),
]

## Комірка 2. Порівняльний прогін

Для кожного тексту й кожної моделі: один виклик, перевірка, чи це валідний
JSON, і вивід поруч, щоб було видно різницю між моделями одразу.

In [ ]:
import json

results = []  # (task_name, text, provider, model, raw_answer, is_valid_json)

for task_name, prompt, texts, tpl in TASKS:
    print(f"\n{'=' * 70}\n{task_name}\n{'=' * 70}")
    for text in texts:
        print(f"\n- {text}")
        for provider, model in MODELS:
            try:
                raw = ask_model(provider, model, prompt, tpl.format(t=text))
                json.loads(raw)  # перевірка валідності
                valid = True
            except json.JSONDecodeError:
                valid = False
            except Exception as err:
                raw = f"ПОМИЛКА ВИКЛИКУ: {err}"
                valid = False
            results.append((task_name, text, provider, model, raw, valid))
            mark = "OK " if valid else "ERR"
            label = f"{provider}:{model}"
            print(f"    [{mark}] {label:<28} {raw}")

## Комірка 3. Підсумок: скільки валідного JSON видала кожна модель

Швидка зведена таблиця, щоб не гортати весь вивід вище.

In [ ]:
from collections import defaultdict

stats = defaultdict(lambda: [0, 0])  # "provider:model" -> [valid, total]
for _, _, provider, model, _, valid in results:
    key = f"{provider}:{model}"
    stats[key][1] += 1
    if valid:
        stats[key][0] += 1

print(f"{'Модель':<28} {'Валідних JSON':<15} {'Разом'}")
for provider, model in MODELS:
    key = f"{provider}:{model}"
    v, t = stats[key]
    print(f"{key:<28} {v:<15} {t}")

## Висновок

Домогтися, щоб усі моделі стабільно давали однакову правильну відповідь —
не проблема: для цього достатньо явно розписати кожне правило й кожен
пограничний випадок, як зроблено в основному файлі
[hennadii_volenbovskyi.ipynb](hennadii_volenbovskyi.ipynb) (там усі три
моделі й усі 45 прогонів збіглися з еталоном). Але цей експеримент із
навмисно простішими, написаними власноруч промптами показує інше: **на
практиці різним моделям потрібні різні промпти**. Той самий промпт, що
чудово працює для однієї моделі, для іншої може виявитися недостатнім.

Це видно й на реальних прогонах вище: `gemini-3.1-pro-preview` сама
здогадалася, що ручний обхід не знижує пріоритет (кейс зі старими курсами
валют), хоча промпт цього прямо не вимагав, а `gemini-2.5-flash-lite` і
`gpt-4.1-nano` — ні. І навпаки, з "плановим обслуговуванням" у Завданні 3
обидві моделі Gemini помилились, а `gpt-4.1-nano` — вгадала. Жодна модель
не виграла в усіх пограничних випадках одразу: кожна по-своєму
"добудовує" ту частину інструкції, яку промпт не проговорює явно, і саме
в цих прогалинах результати різних моделей розходяться.